# Manual Multiclass Logistic Regression Classification

This notebook follows the logistic-regression implementation in `Lab3.ipynb` and the assignment document. It uses the 300-dimensional mean Word2Vec document vectors and the `Category` column from the processed dataset.

Because the dataset has six categories, this notebook uses one multiclass logistic-regression model with a **softmax** output. Softmax produces one probability distribution whose values always add up to `1.0`. No built-in classifier is used.

The categories are imbalanced, so the manual cross-entropy gradient uses square-root class balancing. This improves overall held-out accuracy while still giving minority categories more influence than ordinary unweighted training.

In [2]:
import csv
from pathlib import Path
from sklearn.model_selection import train_test_split

import numpy as np

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

LABEL_PATH = PROJECT_ROOT / "data" / "processed_data" / "preprocessed_research_papers.csv"
VECTOR_PATH = PROJECT_ROOT / "data" / "processed_data" / "document_vectors.npy"
VOCABULARY_PATH = PROJECT_ROOT / "data" / "processed_data" / "word2vec_vocabulary.csv"
WORD_VECTOR_PATH = PROJECT_ROOT / "data" / "processed_data" / "word_vectors.dat"
VECTOR_SIZE = 300

with LABEL_PATH.open("r", encoding="utf-8-sig", newline="") as label_file:
    reader = csv.DictReader(label_file)
    y = np.asarray([row["Category"].strip() for row in reader])

X = np.asarray(np.load(VECTOR_PATH, mmap_mode="r"), dtype=np.float32)

if X.shape[0] != len(y):
    raise ValueError("Vector and label row counts do not match")
if not np.isfinite(X).all():
    raise ValueError("Document vectors contain NaN or infinite values")


# Train-test split (80% train, 20% test)
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

# Standardization using only training data
feature_means = X_train.mean(axis=0, dtype=np.float64).astype(np.float32)
feature_scales = X_train.std(axis=0, dtype=np.float64).astype(np.float32)

# Avoid division by zero
feature_scales[feature_scales == 0] = 1.0

X_train = (X_train - feature_means) / feature_scales
X_test = (X_test - feature_means) / feature_scales

print("Mean Word2Vec feature matrix:", X.shape)
print("Categories:", np.unique(y))
print("Training rows:", len(y_train), "Testing rows:", len(y_test))

Mean Word2Vec feature matrix: (46344, 300)
Categories: ['hpc' 'iot' 'networks' 'nlp' 'security' 'vision']
Training rows: 37075 Testing rows: 9269


In [4]:


def softmax(scores):
    stable_scores = scores - np.max(scores, axis=1, keepdims=True)
    exponentials = np.exp(stable_scores)
    return exponentials / np.sum(exponentials, axis=1, keepdims=True)


def multiclass_cross_entropy(one_hot_labels, probabilities, sample_weights):
    probabilities = np.clip(probabilities, 1e-12, 1.0)
    losses = -np.sum(one_hot_labels * np.log(probabilities), axis=1)
    return np.sum(sample_weights * losses) / np.sum(sample_weights)


def train_multiclass_logistic_regression(
    X, labels, classes, learning_rate=0.1, epochs=1000
):
    sample_count, feature_count = X.shape
    class_count = len(classes)
    weights = np.zeros((feature_count, class_count))
    bias = np.zeros(class_count)
    losses = []

    label_ids = np.searchsorted(classes, labels)
    one_hot_labels = np.zeros((sample_count, class_count))
    one_hot_labels[np.arange(sample_count), label_ids] = 1.0

    class_counts = np.bincount(label_ids, minlength=class_count)
    full_balance_weights = sample_count / (class_count * class_counts)
    # Square-root balancing is a compromise: minority classes matter more,
    # but the largest class is not overwhelmed by noisy minority estimates.
    class_weights = np.sqrt(full_balance_weights)
    sample_weights = class_weights[label_ids]

    for epoch in range(epochs):
        logits = np.dot(X, weights) + bias
        probabilities = softmax(logits)
        weighted_error = sample_weights[:, None] * (probabilities - one_hot_labels)
        dw = (1 / sample_count) * np.dot(X.T, weighted_error)
        db = (1 / sample_count) * np.sum(weighted_error, axis=0)
        weights -= learning_rate * dw
        bias -= learning_rate * db
        losses.append(
            multiclass_cross_entropy(one_hot_labels, probabilities, sample_weights)
        )

    return weights, bias, losses

In [5]:
classes = np.unique(y_train)

# Train one multiclass softmax model, not independent sigmoid models.
multiclass_weights, multiclass_bias, losses = train_multiclass_logistic_regression(
    X_train, y_train, classes
)

test_logits = np.dot(X_test, multiclass_weights) + multiclass_bias
test_probabilities = softmax(test_logits)
predicted_categories = classes[np.argmax(test_probabilities, axis=1)]

print("Trained multiclass softmax model")
print("Probability row sums:", test_probabilities[:3].sum(axis=1))
print("First prediction:", y_test[0], "->", predicted_categories[0])

Trained multiclass softmax model
Probability row sums: [1. 1. 1.]
First prediction: nlp -> networks


In [6]:
from sklearn.metrics import accuracy_score, classification_report

# Accuracy
accuracy = accuracy_score(y_test, predicted_categories)
print(f"Accuracy: {accuracy:.4f}")

# Precision, Recall, F1-score for each class
print(classification_report(
    y_test,
    predicted_categories,
    labels=classes,
    zero_division=0
))

Accuracy: 0.3557
              precision    recall  f1-score   support

         hpc       0.44      0.22      0.30      1333
         iot       0.51      0.44      0.47      3932
    networks       0.27      0.34      0.30      1138
         nlp       0.24      0.46      0.32      1827
    security       0.00      0.00      0.00       185
      vision       0.18      0.08      0.11       854

    accuracy                           0.36      9269
   macro avg       0.27      0.26      0.25      9269
weighted avg       0.38      0.36      0.35      9269



## Try your own research-paper text

The next cell converts your title or abstract into the same averaged Word2Vec document vector used by the model. It then applies multiclass softmax, so the displayed scores form one normalized distribution and sum to `1.0`.

A correct probability calculation does not guarantee a correct category for every text. The current Word2Vec features were trained for only one pass, so the evaluation accuracy is the honest measure of model quality.

In [11]:
from collections import Counter
import re
import pandas as pd

vocabulary_table = pd.read_csv(VOCABULARY_PATH)
word_to_id = dict(zip(vocabulary_table["word"], vocabulary_table["word_id"]))
vector_row_count = max(word_to_id.values()) + 1
word_vectors = np.memmap(
    WORD_VECTOR_PATH,
    mode="r",
    dtype="float32",
    shape=(vector_row_count, VECTOR_SIZE),
)


def text_to_document_vector(text):
    tokens = re.findall(r"[a-z0-9]+(?:[-'][a-z0-9]+)*", text.lower())
    known_vectors = [word_vectors[word_to_id[token]] for token in tokens if token in word_to_id]
    if not known_vectors:
        raise ValueError("No words from this text were found in the Word2Vec vocabulary")
    return np.mean(known_vectors, axis=0).astype(np.float64)


def predict_category(text, show_probabilities=True):
    document_vector = text_to_document_vector(text)
    scaled_vector = (document_vector - feature_means) / feature_scales
    logits = np.dot(scaled_vector, multiclass_weights) + multiclass_bias
    probabilities = softmax(logits.reshape(1, -1))[0]
    assert np.isclose(probabilities.sum(), 1.0)
    predicted_category = classes[np.argmax(probabilities)]

    print("Text:", text)
    print("Predicted category:", predicted_category)
    if show_probabilities:
        print("Softmax probabilities (sum =", f"{probabilities.sum():.4f}):")
        for category, probability in sorted(zip(classes, probabilities), key=lambda item: item[1], reverse=True):
            print(f"  {category}: {probability:.4f}")
    return predicted_category, probabilities

In [21]:
user_text = input("enter user input:")
predict_category(user_text)

Text: artificial intelligence energy efficiency emission reduction industrial systems insights high-pressure air audits flue gas treatment solar plant operations
Predicted category: iot
Softmax probabilities (sum = 1.0000):
  iot: 0.9936
  networks: 0.0048
  vision: 0.0012
  hpc: 0.0003
  security: 0.0001
  nlp: 0.0000


(np.str_('iot'),
 array([2.92888853e-04, 9.93612220e-01, 4.76268577e-03, 2.24868221e-06,
        1.19692192e-04, 1.21026401e-03]))

# BERT comparison model

The following cells use built-in DistilBERT embeddings and
 logistic regression. Run them after the manual model section. The first run downloads the pretrained model, then later runs use cached embeddings.

In [ ]:
import csv
from pathlib import Path

import numpy as np
import torch
from torch.utils.data import DataLoader, Dataset
from transformers import DistilBertForSequenceClassification, DistilBertTokenizerFast
from sklearn.metrics import accuracy_score, classification_report
from sklearn.model_selection import train_test_split

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
DATASET_PATH = PROJECT_ROOT / "data" / "processed_data" / "preprocessed_research_papers.csv"
MODEL_NAME = "distilbert-base-uncased"
MAX_ROWS = 600
MAX_LENGTH = 256
BATCH_SIZE = 8
EPOCHS = 1
LEARNING_RATE = 2e-5

texts = []
labels = []
with DATASET_PATH.open("r", encoding="utf-8-sig", newline="") as data_file:
    reader = csv.DictReader(data_file)
    for row in reader:
        text = (row.get("text") or "").strip()
        label = (row.get("Category") or "").strip()
        if text and label:
            texts.append(text)
            labels.append(label)

labels = np.asarray(labels)
random_generator = np.random.default_rng(42)
selected_indices = []
for category in np.unique(labels):
    category_indices = np.flatnonzero(labels == category)
    random_generator.shuffle(category_indices)
    selected_indices.extend(category_indices[: MAX_ROWS // len(np.unique(labels))])
random_generator.shuffle(selected_indices)
texts = [texts[index] for index in selected_indices]
labels = labels[selected_indices]

classes = np.unique(labels)
label_to_id = {label: index for index, label in enumerate(classes)}
label_ids = np.asarray([label_to_id[label] for label in labels], dtype=np.int64)
train_texts, test_texts, train_labels, test_labels = train_test_split(
    texts,
    label_ids,
    test_size=0.2,
    random_state=42,
    stratify=label_ids,
)

class PaperDataset(Dataset):
    def __init__(self, paper_texts, paper_labels, tokenizer):
        self.texts = paper_texts
        self.labels = paper_labels
        self.tokenizer = tokenizer

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, index):
        encoded = self.tokenizer(
            self.texts[index],
            padding="max_length",
            truncation=True,
            max_length=MAX_LENGTH,
            return_tensors="pt",
        )
        item = {key: value.squeeze(0) for key, value in encoded.items()}
        item["labels"] = torch.tensor(self.labels[index], dtype=torch.long)
        return item


def evaluate_distilbert(model, data_loader, device):
    model.eval()
    predictions = []
    expected = []
    with torch.no_grad():
        for batch in data_loader:
            labels_batch = batch.pop("labels").to(device)
            batch = {key: value.to(device) for key, value in batch.items()}
            logits = model(**batch).logits
            predictions.extend(torch.argmax(logits, dim=1).cpu().numpy())
            expected.extend(labels_batch.cpu().numpy())
    return np.asarray(expected), np.asarray(predictions)


tokenizer = DistilBertTokenizerFast.from_pretrained(MODEL_NAME)
model = DistilBertForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=len(classes),
    id2label={index: label for index, label in enumerate(classes)},
    label2id=label_to_id,
)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

train_loader = DataLoader(
    PaperDataset(train_texts, train_labels, tokenizer),
    batch_size=BATCH_SIZE,
    shuffle=True,
)
test_loader = DataLoader(
    PaperDataset(test_texts, test_labels, tokenizer),
    batch_size=BATCH_SIZE,
)
optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE)

model.train()
for epoch in range(EPOCHS):
    total_loss = 0.0
    for batch in train_loader:
        labels_batch = batch.pop("labels").to(device)
        batch = {key: value.to(device) for key, value in batch.items()}
        optimizer.zero_grad()
        output = model(**batch, labels=labels_batch)
        output.loss.backward()
        optimizer.step()
        total_loss += output.loss.item()
    print(f"DistilBERT epoch {epoch + 1}/{EPOCHS} loss: {total_loss / len(train_loader):.4f}")

test_labels_array, bert_predictions = evaluate_distilbert(model, test_loader, device)
print("DistilBERT device:", device)
print("DistilBERT accuracy:", f"{accuracy_score(test_labels_array, bert_predictions):.4f}")
print(classification_report(
    test_labels_array,
    bert_predictions,
    labels=np.arange(len(classes)),
    target_names=classes,
    zero_division=0,
))

d:\4-1\NLP_Lab\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Some weights of DistilBertForSequenceClassification were not initialized from the model checkpoint at distilbert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight', 'pre_classifier.bias', 'pre_classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


DistilBERT epoch 1/1 loss: 1.7608
DistilBERT device: cpu
DistilBERT accuracy: 0.5000
              precision    recall  f1-score   support

         hpc       0.32      0.60      0.42        20
         iot       0.46      0.65      0.54        20
    networks       0.67      0.10      0.17        20
         nlp       0.77      0.85      0.81        20
    security       0.60      0.30      0.40        20
      vision       0.50      0.50      0.50        20

    accuracy                           0.50       120
   macro avg       0.55      0.50      0.47       120
weighted avg       0.55      0.50      0.47       120



In [3]:
def predict_with_bert(text):
    model.eval()
    encoded = tokenizer(
        text,
        padding="max_length",
        truncation=True,
        max_length=MAX_LENGTH,
        return_tensors="pt",
    )
    encoded = {key: value.to(device) for key, value in encoded.items()}

    with torch.no_grad():
        probabilities = torch.softmax(model(**encoded).logits, dim=1)[0].cpu().numpy()

    prediction = classes[np.argmax(probabilities)]
    print("Text:", text)
    print("Predicted category:", prediction)
    print("Probabilities (sum =", f"{probabilities.sum():.4f}):")
    for category, probability in sorted(
        zip(classes, probabilities),
        key=lambda item: item[1],
        reverse=True,
    ):
        print(f"  {category}: {probability:.4f}")
    return prediction, probabilities

In [7]:
user_text = input("Write a research-paper title or abstract for BERT: ")
predict_with_bert(user_text)

Text: energy saving
Predicted category: iot
Probabilities (sum = 1.0000):
  iot: 0.2069
  hpc: 0.1809
  security: 0.1753
  vision: 0.1556
  nlp: 0.1458
  networks: 0.1356


(np.str_('iot'),
 array([0.1808599 , 0.2069025 , 0.13556705, 0.1457982 , 0.17530386,
        0.15556848], dtype=float32))